# Задание 3. Обучите и протестируйте модель RuT5

За основу возьмем код и данные из [репозитория RuCoLA](https://github.com/RussianNLP/RuCoLA) на github. В коде настроим актуальные версии библиотек и импорты функций, а также будем подбирать свои параметры для обучения, чтобы достичь хорошего результата.

In [ ]:
!git clone https://github.com/RussianNLP/RuCoLA

Cloning into 'RuCoLA'...
remote: Enumerating objects: 76, done.
remote: Counting objects: 100% (76/76), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 76 (delta 31), reused 52 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (76/76), 948.93 KiB | 8.95 MiB/s, done.
Resolving deltas: 100% (31/31), done.


Установим и импортируем необходимые библиотеки.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub")

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
import sys
sys.path.append('/content/RuCoLA/baselines')

In [ ]:
!pip install --upgrade evaluate
!pip install razdel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.4 MB/s eta 0:00:00


In [ ]:
import os
from argparse import ArgumentParser
from functools import partial
from shutil import rmtree

import numpy as np
from evaluate import load
from razdel import tokenize
from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    T5Tokenizer,
    T5ForConditionalGeneration,
    EarlyStoppingCallback
)

from utils import read_splits

Загрузим метрики, которые понадобятся для оценки модели. MCC — это метрика, которая измеряет качество бинарной классификации, выдавая значение от -1 до +1. Она надежно работает даже при сильном дисбалансе классов.

In [ ]:
ACCURACY = load("accuracy")
MCC = metric = load("matthews_correlation")

Зададим параметры обучения:

In [ ]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

N_EPOCHS = 20
learning_rate = 1e-5
weight_decay = 1e-4
batch_size = 128

POS_LABEL = "да"
NEG_LABEL = "нет"

Функция preprocess_examples используется для предобработки:

*   Токенизации примеров
*   Превращения числовых меток в текстовые
*   Токенизации меток
*   Вычисления длины примеров





In [ ]:
def preprocess_examples(examples, tokenizer):
    result = tokenizer(examples["sentence"], padding=False)

    if "acceptable" in examples:
        label_sequences = []
        for label in examples["acceptable"]:
            if label == 1:
                target_sequence = POS_LABEL
            elif label == 0:
                target_sequence = NEG_LABEL
            else:
                raise ValueError("Unknown class label")
            label_sequences.append(target_sequence)
    else:
        label_sequences = ["" for _ in examples["sentence"]]

    result["labels"] = tokenizer(label_sequences, padding=False)["input_ids"]
    result["length"] = [len(list(tokenize(sentence))) for sentence in examples["sentence"]]

    return result

Функция compute_metrics для рассчета метрик:

In [ ]:
def compute_metrics(p, tokenizer):
    # Получаем предсказания
    preds = p.predictions
    if isinstance(preds, tuple):  # для Seq2SeqTrainer иногда возвращается tuple
        preds = preds[0]

    # Если preds это логиты, превращаем в токены (ID)
    if preds.ndim == 3:  # (batch_size, seq_len, vocab_size)
        preds = np.argmax(preds, axis=-1)

    # Защита от выхода за пределы словаря
    preds = np.clip(preds, 0, tokenizer.vocab_size - 1)

    # Декодируем предсказания
    string_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Преобразуем в 0/1
    int_preds = [1 if prediction == POS_LABEL else 0 for prediction in string_preds]

    # Обрабатываем метки
    labels = np.where(p.label_ids != -100, p.label_ids, tokenizer.pad_token_id)
    labels = np.clip(labels, 0, tokenizer.vocab_size - 1)
    string_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    int_labels = []
    for string_label in string_labels:
        if string_label == POS_LABEL:
            int_labels.append(1)
        elif string_label == NEG_LABEL or string_label == "":
            int_labels.append(0)
        else:
            raise ValueError(f"Неизвестная метка: {string_label}")

    # Считаем метрики
    acc_result = ACCURACY.compute(predictions=int_preds, references=int_labels)
    mcc_result = MCC.compute(predictions=int_preds, references=int_labels)

    return {
        "accuracy": acc_result["accuracy"],
        "mcc": mcc_result["matthews_correlation"]
    }

Определим основные параметры для обучения. Поясняю параметры в комментариях:

In [ ]:
model_name = "sberbank-ai/ruT5-base"

tokenizer = T5Tokenizer.from_pretrained(model_name)

splits = read_splits(as_datasets=True)

tokenized_splits = splits.map(
    partial(preprocess_examples, tokenizer=tokenizer),
    batched=True,
    remove_columns=["sentence"],
)

data_collator = DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8)

model = T5ForConditionalGeneration.from_pretrained(model_name)

run_base_dir = f"{model_name}_{learning_rate}_{weight_decay}_{batch_size}"

training_args = Seq2SeqTrainingArguments(
    output_dir=f"checkpoints/{run_base_dir}", #куда сохраняются чекпоинты модели.
    overwrite_output_dir=True, #перезаписывать предыдущие результаты
    eval_strategy="epoch", #оценка модели выполняется в конце каждой эпохи
    per_device_train_batch_size=batch_size, #размер тренировочных батчей
    per_device_eval_batch_size=batch_size, #размер валидационных батчей
    learning_rate=learning_rate, #скорость обучения
    weight_decay=weight_decay, #L2-регуляризация
    num_train_epochs=N_EPOCHS, #кол-во эпох
    logging_strategy="epoch", #логируем ошибку после каждой эпохи
    lr_scheduler_type="cosine", #плавное уменьшение learning rate по косинусной кривой
    warmup_ratio=0.1,
    save_strategy="epoch", #чекпоинт сохраняется в конце каждой эпохи
    save_total_limit=1, #хранить только один лучший чекпоинт.
    fp16=True, #обучение в 16-битном режиме (в 2 раза быстрее и экономнее)
    dataloader_num_workers=2, #два воркера на загрузку данных
    group_by_length=True, #группировка примеров по длине
    report_to="none", #никуда не отправляем логи обучения
    load_best_model_at_end=True, #в конце загружаем лучшую модель
    metric_for_best_model="eval_mcc", #лучшую модель выбираем по метрике mcc
    optim="adamw_torch", #выбираем оптимайзер AdamW
    predict_with_generate=True, #модель будет использовать generate() для предсказаний
    max_grad_norm=1.0 #предотвращает взрыв градиентов.
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_splits["train"],
    eval_dataset=tokenized_splits["dev"],
    compute_metrics=partial(compute_metrics, tokenizer=tokenizer),
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

#trainer.train()

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/1.00M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Map:   0%|          | 0/7869 [00:00<?, ? examples/s]

Map:   0%|          | 0/2787 [00:00<?, ? examples/s]

Map:   0%|          | 0/2789 [00:00<?, ? examples/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/892M [00:00<?, ?B/s]

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Mcc
1,9.976100,3.724727,0.326516,-0.019505
2,2.019800,0.471054,0.674919,-0.013138
3,0.390300,0.326792,0.675637,0.023971
4,0.323900,0.301389,0.674919,0.048246
5,0.304600,0.302397,0.679943,0.087704
6,0.303800,0.300562,0.686042,0.133242
7,0.294900,0.294108,0.693577,0.187386
8,0.290700,0.292274,0.696448,0.206997
9,0.290000,0.290989,0.698601,0.216306
10,0.286500,0.287582,0.704700,0.248507


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=744, training_loss=1.2535572231456797, metrics={'train_runtime': 794.2607, 'train_samples_per_second': 198.147, 'train_steps_per_second': 1.561, 'total_flos': 2187909581045760.0, 'train_loss': 1.2535572231456797, 'epoch': 12.0})

По результатам обучения видно, что модель достигла наилучших результатов на 10 эпохе: MCC около 0.25, это хороший результат. После 10 эпохи валидационная ошибка начинает расти при падающей тренировочной: значит, модель переобучается. EarlyStopper останавливает обучение, сохраняем модель 10 эпохи.

Посмотрим на предсказания модели на валидационном датасете:

In [ ]:
dev_predictions = trainer.predict(test_dataset=tokenized_splits["dev"])
print("Result", dev_predictions.metrics)

Epoch,Training Loss,Validation Loss


Result {'test_loss': 0.28784695267677307, 'test_accuracy': 0.7039827771797632, 'test_mcc': 0.2513208206455225, 'test_runtime': 14.3688, 'test_samples_per_second': 193.962, 'test_steps_per_second': 1.531}


Точность 70%, MCC 0.25 - результаты хорошие, модель обучилась хорошо.